# I1.1 — Heater schedule and energy use (independent)

Exercise card: `I1.1_heater_schedule.md` · 15 min · prerequisites: G1.1; lesson §7 and notebook
`s01_thermal_zone.ipynb` §6 (`simulate`, `piecewise_constant`, `energy_balance`).

**Goal.** Simulate the thermal zone (M01) with a heater power that jumps during the day, and
check the result with an energy balance.

Default M01 parameters ($C = 1.0\cdot10^7$ J/K, $UA = 200$ W/K, illustrative),
$T_{amb} = 0$ °C, $T_0 = 19$ °C, start at 00:00. Heater schedule, repeated every day:
06:00–09:00 → 6 kW; 09:00–17:00 → 2 kW; 17:00–22:00 → 6 kW; 22:00–06:00 → 1 kW.

**How to work.** Run the cells from the top (*Shift+Enter*). Complete the cells marked **TODO** and write your answers in the cells marked *Your answer*. Before the self-check, run **Kernel → Restart Kernel and Run All Cells…**: the notebook must run from top to bottom.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from energy_training.plotting.style import apply_style, degc, hours
from energy_training.simulation import thermal_zone as tz
from energy_training.simulation.inputs import piecewise_constant
from energy_training.utilities import celsius_to_kelvin

apply_style()
H = 3600.0  # s per hour

P = tz.ThermalZoneParams()  # C = 1e7 J/K, UA = 200 W/K (illustrative)
T_AMB = float(celsius_to_kelvin(0.0))  # K
T0 = float(celsius_to_kelvin(19.0))  # K
DAY = [(0, 1000.0), (6, 6000.0), (9, 2000.0), (17, 6000.0), (22, 1000.0)]  # (start hour, W)


def schedule(days=2):
    """Switching times [s] and heater powers [W] for `days` repetitions of DAY."""
    times, vals = [], []
    for d in range(days):
        for hour, q in DAY:
            times.append((24 * d + hour) * H)
            vals.append(q)
    return times, vals


q = piecewise_constant(*schedule())  # heater power [W] as a function of time [s]
t = np.linspace(0, 48 * H, 48 * 12 + 1)  # 5-min output grid [s]

## 1–2. Simulate 48 h

Simulate with `tz.simulate` on the output grid `t` (argument `t_eval`). The signal `q` knows its
switching times, so the solver restarts at each jump. Store the result in `res`
(fields `res.t`, `res.T`, `res.Q_flow`, `res.T_amb`).

In [ ]:
# TODO: res = tz.simulate(...)

## 3. Plot

Zone temperature (°C) and heater power (kW) against time (h), one above the other.

In [ ]:
# TODO: plot temperature [degC] and heater power [kW] against time [h]

## 4. Heater energy of day 2

Compute it by hand from the schedule, then from the simulation output (`np.trapezoid` of the heater
power over day 2). Store both in kWh as `E_day2_hand_kWh` and `E_day2_sim_kWh`.

*Your hand calculation:*

In [ ]:
# TODO: E_day2_hand_kWh = ...
# TODO: E_day2_sim_kWh = ...

## 5. Energy balance

Evaluate `tz.energy_balance(res, P)` for the 48 h and the residual relative to the supplied energy
(`residual_rel`). Explain its size. Compare it with the residual in notebook §6 and test the
hypothesis you wrote there.

In [ ]:
# TODO: eb = tz.energy_balance(...); residual_rel = ...

*Your answer:*

## 6. Comfort on day 2

Minimum and maximum zone temperature on day 2 (`T_min_day2_C`, `T_max_day2_C`). Is the zone
comfortable (≥ 19 °C) during occupied hours (06:00–22:00)?

In [ ]:
# TODO: T_min_day2_C = ..., T_max_day2_C = ...

*Your answer:*

## Self-check

Expected outcome of the card: heater energy of day 2 from the simulation within 1 % of the hand
value, energy-balance residual below 1 % of the supplied energy.

In [ ]:
missing = [n for n in ("E_day2_hand_kWh", "E_day2_sim_kWh", "residual_rel") if n not in globals()]
assert not missing, f"complete steps 4 and 5 first: {', '.join(missing)} not defined"
assert abs(E_day2_sim_kWh / E_day2_hand_kWh - 1) < 0.01, "simulated day-2 energy is off"
assert residual_rel < 0.01, "energy-balance residual above 1 %: check the simulation and grid"
print("Self-check passed.")

## Extension: night power

Find the smallest constant night power (22:00–06:00) for which the zone temperature **at 06:00 on
day 2** is ≥ 18 °C. Then check: is the zone also ≥ 18 °C during the rest of day 2? (Hint: look at
the 09:00–17:00 period at 2 kW.) Solve it twice: by trial simulations, and with the analytic
piecewise solution (`tz.analytic_piecewise`) plus `scipy.optimize.brentq`.

In [ ]:
# TODO (optional)

*Your answer:*